In [1]:
import sys
import os

# Add project root to Python path
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.file_processor import AdvancedFileProcessor
from utils.vector_db import VectorDatabase
from utils.rag_system import EnhancedRAGSystem
import config

# Initialize components
processor = AdvancedFileProcessor()
vector_db = VectorDatabase()
rag_system = EnhancedRAGSystem(vector_db, project_root + config.LLM_MODEL)

!!!!!!!!!!!!megablocks not available, using torch.matmul instead
<All keys matched successfully>
llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from c:\Users\sheref.abolmagd_doct\Desktop\AI NLP/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
lla

In [2]:
import os
from tqdm.notebook import tqdm

all_chunks = []
publications_dir = os.path.normpath("../publications")  # Normalize path

for root, dirs, files in os.walk(publications_dir):
    for file in tqdm(files, desc="Processing files"):
        file_path = os.path.normpath(os.path.join(root, file))
        chunks = processor.process_file(file_path)
        if chunks:
            all_chunks.extend(chunks)

if all_chunks:
    vector_db.add_documents(all_chunks)
    print(f"Successfully processed {len(all_chunks)} chunks from {len(files)} files")
else:
    print("No documents were processed. Check:")
    print(f"- Directory exists: {os.path.exists(publications_dir)}")
    print(f"- Files found: {len(files)}")
    print(f"- First file path: {os.path.join(root, files[0]) if files else 'N/A'}")

Processing files:   0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\sheref.abolmagd_doct\Desktop\AI NLP\venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\sheref.abolmagd_doct\Desktop\AI NLP\venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\sheref.abolmagd_doct\Desktop\AI NLP\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\sheref.abolmagd_doct\Desktop\AI NLP\venv\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Wedding budget'!$A:$K.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


Successfully processed 441 chunks from 10 files


In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

# Create UI elements
question_input = widgets.Textarea(
    value='',
    placeholder='Ask about Dr. X\'s research...',
    description='Question:',
    layout={'width': '80%'}
)

submit_button = widgets.Button(description="Submit")
output_area = widgets.Output()

def on_submit(b):
    with output_area:
        output_area.clear_output()
        question = question_input.value
        if question:
            print(f"Q: {question}")
            start_time = time.time()
            print("Generating Answer...")
            answer = rag_system.ask(question)
            duration = time.time() - start_time
            print(f"A: {answer}")
            print(f"\n(Generated in {duration:.2f} seconds)")
        else:
            print("Please enter a question")

submit_button.on_click(on_submit)

display(question_input, submit_button, output_area)

Textarea(value='', description='Question:', layout=Layout(width='80%'), placeholder="Ask about Dr. X's researc…

Button(description='Submit', style=ButtonStyle())

Output()

In [ ]:
from utils.translator import PublicationTranslator

# Initialize translator with same LLM
translator = PublicationTranslator(project_root + config.LLM_MODEL)

# Select a sample chunk to translate (e.g., first chunk)
sample_chunk = all_chunks[20]
print(f"Original text ({len(sample_chunk['text'])} chars):\n{sample_chunk['text'][:200]}...\n")

# Translate to Arabic
translation = translator.translate(sample_chunk['text'], "Arabic")
print(f"Translated text ({len(translation['translated_text'])} chars):\n{translation['translated_text'][:200]}...\n")
print("Translation metrics:")
print(f"- Source language: {translation['source_lang']}")
print(f"- Speed: {translation['tokens_per_sec']:.1f} tokens/sec")
print(f"- Duration: {translation['time_sec']:.2f} seconds")

In [ ]:
from utils.summarizer import PublicationSummarizer

# Initialize summarizer
summarizer = PublicationSummarizer(project_root + config.LLM_MODEL)

# Combine first 3 chunks as sample document
sample_text = " ".join(chunk['text'] for chunk in all_chunks[:3])

# Generate different summary types
strategies = ["key_points", "technical", "layman"]
for strategy in strategies:
    summary = summarizer.summarize(sample_text, strategy)
    print(f"\n=== {strategy.upper()} SUMMARY ===")
    print(summary['summary'])
    print(f"\nROUGE Scores:")
    print(f"- ROUGE-1: {summary['rouge_scores']['rouge1'].fmeasure:.3f}")
    print(f"- ROUGE-2: {summary['rouge_scores']['rouge2'].fmeasure:.3f}")
    print(f"- Speed: {summary['metrics']['tokens_per_sec']:.1f} tokens/sec")